# Model Insights, Threshold Selection and Deployment

**Course:** IS6400 Business Data Analytics — Group Project (6-person team)
**Author of this section:** Mengyang Cao

This notebook contains **only the section I was responsible for** within the group project on
credit card customer churn prediction. The full pipeline (data cleaning, feature engineering,
model training and comparison across Logistic Regression / Random Forest / GBDT / Transformer)
was produced by the team; the parts below are mine:

1. **Model insights** — interpreting Gradient Boosting feature importances into business drivers
2. **Threshold selection** — precision-recall analysis to choose the operating threshold
3. **Deployment** — scoring the full customer base and producing the at-risk customer list

> Note: cells assume the upstream objects (`gb_best`, `X`, `X_train_final`, `X_test_final`,
> `y_test`, `df_final`, `df_cleaned`, `scaler`, `df_original`) produced by the team's earlier
> pipeline. The raw dataset is not redistributed here.

### 4.1 Model Insights

In [ ]:
feature_names = X.columns # retrieve all feature names from the full feature set

print("--- Feature Importance Analysis ---")
print("\nGradient Boosting Feature Importances:")
gb_importances = pd.Series(gb_best.feature_importances_, index=feature_names).sort_values(ascending=False)
print(gb_importances.head(20)) # top 20

plt.figure(figsize=(12, 8))
sns.barplot(x=gb_importances.head(20).values, y=gb_importances.head(20).index, palette='magma')
plt.title('Top 20 Feature Importances (Gradient Boosting)')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

We can clearly see that ***trans_ct***  standsout as overwhelmingly the most important, with an importance score significantly higher than any other feature, suggesting it's a primary driver of the models predictions. Following trans_ct, **revolving_bal** and **trans_amt** are also highly impactful features. Other features like *ct_chg*, *relationships*, *avg_trans_amt*, and *amt_chg* show moderate importance, while features towards the bottom of the list, such as *tenure_mouths*, *credit_usage_efficiency*, and *limit*, contribute minimally to the model's decision-making process.

#### 4.1-1 Optimal Feature Selection

In [ ]:
top_15_features = gb_importances.head(15).index.tolist()
print(f"Top 15 features: {top_15_features}")

In [ ]:
X_train_top15 = X_train_final[top_15_features]
X_test_top15 = X_test_final[top_15_features]

print(f"\nShape of X_train_top15: {X_train_top15.shape}")
print(f"Shape of X_test_top15: {X_test_top15.shape}")

In [ ]:
# select a piece of set to train
X_tr_top, X_val_top, y_tr, y_val = train_test_split(
    X_train_top15,
    y_train,
    test_size=0.3,
    random_state=50,
    stratify=y_train
)

print("Train for GB:", X_tr_top.shape, "Valid for GB:", X_val_top.shape)

In [ ]:
#choose the best AUC
gb_candidates = [
    # faster,weaker
    {"n_estimators": 150, "learning_rate": 0.08, "max_depth": 2, "subsample": 0.9},
    # more balance
    {"n_estimators": 250, "learning_rate": 0.05, "max_depth": 3, "subsample": 0.85},
    # smoother
    {"n_estimators": 300, "learning_rate": 0.03, "max_depth": 3, "subsample": 0.85},
]

best_auc = -1
best_params_top15 = None

for params in gb_candidates:
    gb_top = GradientBoostingClassifier(
        random_state=50,
        **params
    )
    gb_top.fit(X_tr, y_tr)
    val_prob_top = gb_top.predict_proba(X_val)[:, 1]
    val_auc = roc_auc_score(y_val, val_prob_top)
    print(f"Params {params} -> val AUC = {val_auc:.4f}")
    if val_auc > best_auc:
        best_auc = val_auc
        best_params_top15 = params

print("\n Best params for GB on our validation set:", best_params_top15)

In [ ]:
gb_best_top = GradientBoostingClassifier(
    random_state=50,
    **best_params
)
gb_best_top.fit(X_train_top15, y_train)

gb_pred_top = gb_best_top.predict(X_test_top15)
gb_prob_top = gb_best_top.predict_proba(X_test_top15)[:, 1]

In [ ]:
print("\nOptimal Features Evaluation for Gradient Boosting:")
print(f"Accuracy : {accuracy_score(y_test, gb_pred_top):.4f}")
print(f"Precision: {precision_score(y_test, gb_pred_top):.4f}")
print(f"Recall   : {recall_score(y_test, gb_pred_top):.4f}")
print(f"F1-score : {f1_score(y_test, gb_pred_top):.4f}")
print(f"ROC AUC  : {roc_auc_score(y_test, gb_prob_top):.4f}")


# Confusion Matrix
cm = confusion_matrix(y_test, gb_pred_top)
plt.figure(figsize=(8, 6))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Confusion Matrix: Gradient Boosting')
plt.colorbar()
classes = ['Existing Customer', 'Attrited Customer']
tick_marks = np.arange(len(classes))
plt.xticks(tick_marks, classes, rotation=45)
plt.yticks(tick_marks, classes)
fmt = 'd'
thresh = cm.max() / 2.
for i, j in np.ndindex(cm.shape):
    plt.text(j, i, format(cm[i, j], fmt),
             horizontalalignment="center",
             color="white" if cm[i, j] > thresh else "black")
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.tight_layout()
plt.show()

# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, gb_prob_top)
roc_auc = auc(fpr, tpr)
plt.figure()
plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve: Gradient Boosting')
plt.legend(loc="lower right")
plt.show()

### 4.2 Identifying optimal categorisation threshold

In [ ]:
# compute the precision-recall curve
y_scores = gb_best.predict_proba(X_test_final)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_test, y_scores)

# plot Precision & Recall against threshold
plt.figure(figsize=(10, 7))
plt.plot(thresholds, precision[:-1], "b--", label="Precision")
plt.plot(thresholds, recall[:-1], "g-", label="Recall")
plt.xlabel("Threshold")
plt.ylabel("Score")
plt.title("Precision-Recall vs Threshold")
plt.legend(loc="lower left")
plt.ylim([0, 1])
plt.grid(True)
plt.show()

# compute F1-score and plot it
f1_scores = 2 * precision * recall / (precision + recall)
plt.figure(figsize=(10, 7))
plt.plot(thresholds, f1_scores[:-1], "r-", label="F1-score")
plt.xlabel("Threshold")
plt.ylabel("F1-score")
plt.title("F1-score vs Threshold")
plt.grid(True)
plt.legend()
plt.show()

# locate the optimal threshold (corrected version)
optimal_idx = np.argmax(f1_scores[:-1])
optimal_threshold = thresholds[optimal_idx]
print(f"Best Threshold (max F1): {optimal_threshold:.4f}")
print(f"Corresponding F1: {f1_scores[:-1][optimal_idx]:.4f}")

### 4.3 Model Deployment

In [ ]:
# prepare the full feature set (X_full) for prediction
X_full = df_final.drop('churn_flag', axis=1)

numerical_cols_for_model = [col for col in X_full.select_dtypes(include=np.number).columns.tolist() if col != 'client_id']
non_numerical_cols_for_model = [col for col in X_full.select_dtypes(exclude=np.number).columns.tolist() if col != 'client_id']

X_full_scaled_numerical = scaler.transform(X_full[numerical_cols_for_model])
X_full_processed = pd.DataFrame(X_full_scaled_numerical, columns=numerical_cols_for_model, index=X_full.index)
X_full_processed = pd.concat([X_full_processed, X_full[non_numerical_cols_for_model]], axis=1)
X_full_processed = X_full_processed[X_train_final.columns]

# predicted churn probability for the entire dataset
df_final_for_dashboard = df_final.copy() # Make a copy to add new columns

df_final_for_dashboard['predicted_churn_probability'] = gb_best.predict_proba(X_full_processed)[:, 1]

churn_risk_threshold = 0.44

# assign predicted churn class using the chosen threshold
df_final_for_dashboard['predicted_churn_class'] = (df_final_for_dashboard['predicted_churn_probability'] >= churn_risk_threshold)

# flag customers currently at risk but not yet churned
df_final_for_dashboard['is_currently_at_risk'] = (
    (df_final_for_dashboard['predicted_churn_probability'] >= churn_risk_threshold) &
    (df_final_for_dashboard['churn_flag'] == False) # Only include actual non-churners
)

In [ ]:
num_at_risk_customers = df_final_for_dashboard['is_currently_at_risk'].sum()

print(f"Number of currently at-risk customers: {num_at_risk_customers}")

total_customers = len(df_final_for_dashboard)
percentage_at_risk = (num_at_risk_customers / total_customers) * 100
print(f"Percentage of currently at-risk customers: {percentage_at_risk:.2f}%")

# columns to drop from the dashboard dataframe
# encoded columns to drop
encoded_cols_to_drop = [
    'education_encoded',
    'income_encoded',
    'gender_m',
    'marital_married',
    'marital_single',
    'marital_unknown',
    'card_type_gold',
    'card_type_platinum',
    'card_type_silver'
]

# original categorical columns to keep / re-join
# (these are the original columns behind the encodings)
original_categorical_cols = [
    'gender',
    'edu',
    'marital',
    'income',
    'card_type'
]

df_dashboard_final_display = df_final_for_dashboard.copy()
df_dashboard_final_display = df_dashboard_final_display.drop(columns=encoded_cols_to_drop, errors='ignore')

original_data_for_merge = df_cleaned[['client_id'] + original_categorical_cols]

df_dashboard_final_display = pd.merge(
    df_dashboard_final_display,
    original_data_for_merge,
    on='client_id',
    how='left' 
)

# reorder columns (optional, improves readability) ---
current_cols = df_dashboard_final_display.columns.tolist()

# define the desired order
new_prediction_cols = [
    'predicted_churn_probability',
    'predicted_churn_class',
    'is_currently_at_risk'
]

# keep only original dataset column names
initial_cols = [col for col in df_original.columns if col not in new_prediction_cols and col not in encoded_cols_to_drop] # All original columns except churn_flag and those that were encoded

# final column order for the dashboard dataframe
desired_order = [
    'client_id', 'churn_flag', 'age', 'gender', 'dependents', 'edu', 'marital', 'income',
    'card_type', 'tenure_months', 'relationships', 'inactive_months',
    'contact_count', 'limit', 'revolving_bal', 'open_to_buy', 'amt_chg_q4_q1',
    'trans_amt', 'trans_ct', 'ct_chg_q4_q1', 'util_ratio', # Original numerical/categorical
    'trans_amt_per_month', 'trans_ct_per_month', 'avg_trans_amt', # Engineered numerical
    'balance_to_limit_ratio', 'credit_usage_efficiency', 'credit_growth_rate', # Engineered numerical
    'predicted_churn_probability', 'predicted_churn_class', 'is_currently_at_risk' # New model outputs
]

# keep only columns actually present in the dataframe
# some intermediate columns may have been dropped or transformed
final_col_order = [col for col in desired_order if col in df_dashboard_final_display.columns]

df_dashboard_final_display = df_dashboard_final_display[final_col_order]

# inspect the final dataframe
print("\n--- Info for Final DataFrame ---")
df_dashboard_final_display.info()

In [ ]:
# show the first 20 users of the dashboard dataframe
# -------------------------------------------------
N = 20
display_df = df_dashboard_final_display.head(N).copy()


# highlight high-risk customers with pandas Styler
styled_df = (display_df.style
    # highlight: high-risk customers get a red row background
    .apply(lambda row: ['background: #ffcccc; font-weight: bold' 
                       if row['predicted_churn_class'] 
                       else 'background: white' for _ in row], axis=1)
    
    # format numeric columns
    .format({
        'predicted_churn_probability': '{:.4f}',
        'limit': '${:,.0f}',
        'open_to_buy': '${:,.0f}',
        'trans_amt': '${:,.0f}',
        'util_ratio': '{:.3f}',
        'amt_chg': '{:.3f}',
        'ct_chg': '{:.3f}'
    })
    
    # set column widths (optional)
    .set_properties(**{'text-align': 'center'})
    .set_table_styles([{
        'selector': 'th',
        'props': [('background-color', '#4CAF50'), ('color', 'white'), ('font-weight', 'bold')]
    }])
)

# display
display(styled_df)

In [ ]:
# show the top 20 highest-risk customers
# -------------------------------------------------
N = 20
display_df = df_dashboard_final_display.copy()

# sort by predicted churn probability, descending
display_df = display_df.sort_values('predicted_churn_probability', ascending=False)

# red marks the high-risk customers requiring attention
print(f"Top {N} customers by predicted churn probability:\n")
display(display_df.head(N).style
        .apply(lambda row: ['background: #ffcccc' if row['predicted_churn_class'] 
                           else 'background: white' for _ in row], axis=1)
        .format({
            'predicted_churn_probability': '{:.4f}',
            'limit': '${:,.0f}',
            'open_to_buy': '${:,.0f}',
            'trans_amt': '${:,.0f}'
        }))